In [13]:
import sys
from types import MethodType

sys.path.append(r"C:\Users\AlexanderZirn\Documents\GIT\nRTD\lib")
from pathlib import Path
import argparse

import numpy as np
import torch
import torch.utils
import torch.utils.data
import wandb
import lightning.pytorch as pl
from lightning.pytorch.loggers.wandb import WandbLogger
from lightning.pytorch.callbacks import LearningRateMonitor

torch.set_default_dtype(torch.float64)

from nRTD import RTDModule, RTDDataModule
#from SweepRunner import Sweeper
import matplotlib.pyplot as plt

plt.style.use("ICIWstyle")

In [14]:
def log_plots(data_module: RTDDataModule, model: RTDModule, index=0, Es_to_plot=[0]):
    pred_y, pred_t = model(data_module.x, data_module.t_in)
    pred_t = pred_t.detach().numpy().squeeze()
    pred_y = pred_y.detach().numpy().squeeze()

    fig = plt.figure()
    index = 0
    plt.plot(
        data_module.t_in[index],
        data_module.x[index].squeeze(),
        ls="--",
    )
    plt.plot(
        data_module.t_in[index + 2],
        data_module.x[index + 2].squeeze(),
        ls="--",
    )
    plt.plot(
        data_module.t_out[index],
        data_module.y[index].squeeze(),
        "o",
        c="C0",
        markersize=0.5,
    )
    plt.plot(
        data_module.t_out[index + 2],
        data_module.y[index + 2].squeeze(),
        "o",
        c="C1",
        markersize=0.5,
    )
    plt.plot(
        data_module.t_in[index + 1],
        data_module.x[index + 1].squeeze(),
        ls="--",
        c="C2",
    )
    plt.plot(
        data_module.t_in[index + 3],
        data_module.x[index + 3].squeeze(),
        ls="--",
        c="C3",
    )
    plt.plot(
        data_module.t_out[index + 1],
        data_module.y[index + 1].squeeze(),
        "o",
        c="C2",
        markersize=0.5,
    )
    plt.plot(
        data_module.t_out[index + 3],
        data_module.y[index + 3].squeeze(),
        "o",
        c="C3",
        markersize=0.5,
    )

    plt.plot(pred_t[index, :], pred_y[index, :], c="C0")
    plt.plot(pred_t[index + 2].squeeze(), pred_y[index + 2].squeeze(), c="C1")
    plt.plot(pred_t[index + 1, :], pred_y[index + 1, :], c="C2")
    plt.plot(pred_t[index + 3].squeeze(), pred_y[index + 3].squeeze(), c="C3")
    plt.xlabel("t / s")
    plt.ylabel("x / 1")
    plt.twinx()
    ax = plt.gca()
    plt.ylabel("$E \;/\; s^{-1}$")
    my_colors = ["darkmagenta", "goldenrod", "firebrick", "darkcyan"]
    for E_index in Es_to_plot:
        ax.plot(
            model.net.conv_layers[E_index].t_kernel.detach().numpy(),
            model.net.E[E_index],
            color=my_colors[E_index],
        )
    wandb.log({f"RTD_Plot_img": wandb.Image(fig)})
    wandb.log({f"RTD_Plot_fig": fig})

In [ ]:
### Constants
# givens
t_out = (0.0, 120.0)
n_out = int(4 * (t_out[1] - t_out[0]) + 1)
delta_t_out = (t_out[1] - t_out[0]) / (n_out - 1)

# # the capillary times
# # FBA_18032025_300ml-min_Analytik_MS
# switching_periods = np.array(
#     [
#         5 * 60 + 0.03,
#         5 * 60 + 0.27,
#         5 * 60 + 0.27,
#         5 * 60 + 0.28,
#         5 * 60 + 0.18,
#         5 * 60 + 0.26,
#         5 * 60 + 0.33,
#         5 * 60 + 0.27,
#         5 * 60 + 0.18,
#         5 * 60 + 0.05,
#     ]
# )
# switching_times_cap = np.flip(np.cumsum(-switching_periods))
# switching_times_cap = np.append(switching_times_cap, 0) - 1

# the piping times
t_halfperiod = t_out[1] - t_out[0]

# automated_switching_times = np.full((20,), t_halfperiod)
# switching_times_reac_inlet = np.insert(automated_switching_times, 0, 7.4 - 1.0)
# switching_times_reac_inlet = np.cumsum(switching_times_reac_inlet)
# switching_times_reac_outlet = np.insert(automated_switching_times, 0, 7.5 - 1.0)
# switching_times_reac_outlet = np.cumsum(switching_times_reac_outlet)
# switching_times_total_system = np.insert(automated_switching_times, 0, 7.7 - 1.0)
# switching_times_total_system = np.cumsum(switching_times_total_system)

t_kernels = [
    #(0.0, 30.0),  # piping to reactor inlet
    #(0.0, 40.0),  # reactor
    (0.0, 15.0)#,  # piping after reactor outlet
    #(0.0, 30.0),  # capillary
]

n_kernels = list(
    map(
        lambda t_kernel: int(((t_kernel[1] - t_kernel[0]) / delta_t_out) + 1), t_kernels
    )
)

t_in = (0, t_out[1] - sum([t_kernel[1] for t_kernel in t_kernels]))
n_in = int(((t_in[1] - t_in[0]) / delta_t_out) + 1)

wandb.init(project="HSA_MGA_nRTD", entity="ice_ulm")






switching_times = np.full((10,), t_halfperiod) + np.array([0.05,
                                                          0.18,
                                                          0.27,
                                                          0.33,
                                                          0.26,
                                                          0.18,
                                                          0.28,
                                                          0.27,
                                                          0.27,
                                                          0.03])# 24 half-periods

lr-Adam,▁
trainer/global_step,▁
lr-Adam,0.004
trainer/global_step,0


In [16]:
import sys

sys.path.append(r"C:\Users\AlexanderZirn\Documents\GIT\nRTD\lib")
from pathlib import Path
from nRTD.rtd_fitting import RTDDataModule

data_total_system = RTDDataModule(
    batch_size= 32, # 2 temperatures, 2 half-periods, 4 different capillary positions
    data_file=Path(
        r"C:\Users\AlexanderZirn\Documents\GIT\nRTD\Experiments\AZA_001\D - downstream\AZA-E-130625_250ml-min_2bar_250_TotalSystem.npz"
    ),
    switching_times=switching_times,
    t_range_in=(0, 10), # kernel discretization
    n_in=41, # kernel
    t_range_out=(0, 120), # time range of output time series
    n_out=481,  # number of points in output time series in s (4 MS measurements per second)
    switch_delay=1.0,
)

In [ ]:
# data_cap = RTDDataModule(
#     batch_size=32,
#     data_file=Path(
#         r"D:\Users\Hannes\Documents\Python Code\nRTD\Experiments\HSA_001\data\FBA-E-18032025_300ml-min_Analytik_MS.npz"
#     ),
#     switching_times=switching_times_cap,
#     t_range_in=t_in,
#     n_in=n_in,
#     t_range_out=t_out,
#     n_out=n_out,
#     switch_delay=1.0,
# )


# data_reac_inlet = RTDDataModule(
#     batch_size=32,
#     data_file=Path(
#         r"D:\Users\Hannes\Documents\Python Code\nRTD\Experiments\HSA_002\data\MGA-E-11032025_30ml-min_1atm_ReactorInlet_MS.npz"
#     ),
#     switching_times=switching_times_reac_inlet,
#     t_range_in=t_in,
#     n_in=n_in,
#     t_range_out=t_out,
#     n_out=n_out,
#     switch_delay=1.0,
# )

# data_reac_outlet = RTDDataModule(
#     batch_size=32,
#     data_file=Path(
#         r"D:\Users\Hannes\Documents\Python Code\nRTD\Experiments\HSA_003\data\MGA-E-11032025_30ml-min_1atm_ReactorOutlet_MS.npz"
#     ),
#     switching_times=switching_times_reac_outlet,
#     t_range_in=t_in,
#     n_in=n_in,
#     t_range_out=t_out,
#     n_out=n_out,
#     switch_delay=1.0,
# )

# data_total_system = RTDDataModule(
#     batch_size=32,
#     data_file=Path(
#         r"D:\Users\Hannes\Documents\Python Code\nRTD\Experiments\HSA_004\data\MGA-E-10032025_120ml-min_4bar_TotalSystem.npz"
#     ),
#     switching_times=switching_times_total_system,
#     t_range_in=t_in,
#     n_in=n_in,
#     t_range_out=t_out,
#     n_out=n_out,
#     switch_delay=1.0,
# )


model = RTDModule(
    kernel_sizes = n_kernels,
    kernel_times = t_kernels,
    learning_rate = 4e-3,
    use_scheduler = False,
    scheduler_kwargs = {"factor": 0.6, "patience": 2000},
)

In [21]:
my_logger = WandbLogger(log_model=True)
lr_callback = LearningRateMonitor()

max_epochs = 20_000

trainer = pl.Trainer(
    accelerator="cpu",
    max_epochs=max_epochs,
    enable_progress_bar=True,
    logger=my_logger,
    callbacks=[lr_callback],
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [20]:
for conv_layer in model.net.conv_layers:
    conv_layer.set_to_direct_response()
    conv_layer.freeze()


# model.net.conv_layers[3].unfreeze()
# trainer.fit(model, data_cap)

# log_plots(
#     data_module=data_cap,
#     model=model,
#     index=0,
#     Es_to_plot=[3],
# )

# model.net.conv_layers[3].freeze()
# model.net.conv_layers[0].unfreeze()
# model.configure_optimizers()
# trainer = pl.Trainer(
#     accelerator="cpu",
#     max_epochs=max_epochs,
#     enable_progress_bar=True,
#     logger=my_logger,
#     callbacks=[lr_callback],
# )
# trainer.fit(model, data_reac_inlet)

# log_plots(
#     data_module=data_reac_inlet,
#     model=model,
#     index=0,
#     Es_to_plot=[0, 3],
# )

# model.net.conv_layers[0].freeze()
# model.net.conv_layers[1].unfreeze()
# model.configure_optimizers()
# trainer = pl.Trainer(
#     accelerator="cpu",
#     max_epochs=max_epochs,
#     enable_progress_bar=True,
#     logger=my_logger,
#     callbacks=[lr_callback],
# )
# trainer.fit(model, data_reac_outlet)

# log_plots(
#     data_module=data_reac_outlet,
#     model=model,
#     index=0,
#     Es_to_plot=[0, 1, 3],
# )

#model.net.conv_layers[1].freeze()
model.net.conv_layers[2].unfreeze()
model.configure_optimizers()
trainer = pl.Trainer(
    accelerator="cpu",
    max_epochs=max_epochs,
    enable_progress_bar=True,
    logger=my_logger,
    callbacks=[lr_callback],
)
trainer.fit(model, data_total_system)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name | Type   | Params | Mode 
----------------------------------------
0 | net  | RTDNet | 464    | train
----------------------------------------
61        Trainable params
403       Non-trainable params
464       Total params
0.002     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode


Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s] 

RuntimeError: The size of tensor a (501) must match the size of tensor b (481) at non-singleton dimension 2

In [ ]:
log_plots(
    data_module=data_total_system,
    model=model,
    index=0,
    Es_to_plot=[0, 1, 2, 3],
)

plt.show()